# Week 2: Raster Preprocessing (Clip, Reproject, Resample)

> **Milestone:** Rasterio: Clip, reproject, resample rasters
> **Deliverable:** Notebook 02 complete

*From PROJECT_BRIEF.md - Phase 1: Foundation (Weeks 1-4)*

In [ ]:
# Import libraries
import geopandas as gpd
import rasterio
from rasterio.plot import show
from rasterio.mask import mask
import numpy as np
import os

# Set working directory
wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)

# Define our region of interest (Southern Africa focus)
from shapely.geometry import box
roi = box(12.0, -20.0, 22.0, -10.0)
roi_gdf = gpd.GeoDataFrame({'geometry': [roi]}, crs='EPSG:4326')

# Function to clip and reproject raster to ROI
def clip_reproject_raster(input_path, output_path, roi_gdf, target_crs='EPSG:4326'):
    with rasterio.open(input_path) as src:
        if roi_gdf.crs != src.crs:
            roi_projected = roi_gdf.to_crs(src.crs)
        else:
            roi_projected = roi_gdf
        out_image, out_transform = mask(src, roi_projected.geometry, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            'height': out_image.shape[1],
            'width': out_image.shape[2],
            'transform': out_transform,
            'crs': target_crs
        })
        with rasterio.open(output_path, 'w', **out_meta) as dest:
            dest.write(out_image)
    return output_path

print('clip_reproject_raster function defined')

In [ ]:
# Clip and reproject all rasters to ROI
raw_dir = 'data/raw'
processed_dir = 'data/processed'
os.makedirs(processed_dir, exist_ok=True)

# List of rasters to process
rasters_to_process = [
    'solar_irradiance.tif',
    'water_stress.tif',
    'population_density.tif'
]

for raster_name in rasters_to_process:
    input_path = os.path.join(raw_dir, raster_name)
    output_path = os.path.join(processed_dir, f'clipped_{raster_name}')

    try:
        result = clip_reproject_raster(input_path, output_path, roi_gdf)
        print(f'✅ Processed {raster_name}')
    except Exception as e:
        print(f'❌ Error processing {raster_name}: {e}')

print('\nAll rasters processed!')